# Feature Exploration Update — Sample Frames, Feature Viz, and New Directions

This notebook extends `feature_extraction.ipynb` for the project update:

1. A per-class sample image + feature visualization panel (raw frame, HOG, YOLO detections, Hough/vanishing-point, road segmentation) for all 5 maneuver classes.
2. t-SNE and UMAP projections alongside the existing PCA, to get a second (nonlinear) read on how separable each feature set is by maneuver class.
3. A new complex feature: a dense CNN embedding pulled from the YOLOv8s backbone (not just detection counts), using `ultralytics`' built-in `embed` support.
4. A written summary of what we're seeing and where to focus tinkering effort next.

Run this with the `281-s2-group2` kernel — it needs `standard_e2e`, `ultralytics`, and the feature arrays already produced by `feature_extraction.ipynb`.

> `pip install umap-learn` if you haven't already (only new dependency vs. the existing environment.yml).

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import json
import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import Counter
from standard_e2e import Modality

TRAIN_DIR     = '../data/processed/waymo_e2e/training/'
MANIFEST_PATH = '../data/processed/waymo_e2e/train_manifest.json'
FEATURES_DIR  = '../data/processed/waymo_e2e/features/'

CLASSES = ['straight', 'left-turn', 'right-turn', 'lane-change-left', 'lane-change-right']
COLORS = {
    'straight': 'steelblue', 'left-turn': 'tomato', 'right-turn': 'green',
    'lane-change-left': 'purple', 'lane-change-right': 'orange',
}

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

examples = {}
for seq_id, entry in manifest.items():
    label = entry['label']
    if label in CLASSES and label not in examples:
        data = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
        examples[label] = np.array(data['_modality_data'].item()[Modality.CAMERAS])
    if len(examples) == len(CLASSES):
        break

print(Counter(v['label'] for v in manifest.values()))

## 1. Per-class sample image + feature visualization panel

Five rows per class: raw frame, HOG, YOLO detections, Hough lines + vanishing point, road segmentation.
This is the literal deliverable for the project update — reuses the exact feature functions from
`feature_extraction.ipynb` so the visuals stay consistent with what's already been reviewed.

In [ ]:
from skimage.feature import hog
from ultralytics import YOLO

HOG_PARAMS = {'orientations': 9, 'pixels_per_cell': (16, 16), 'cells_per_block': (2, 2), 'channel_axis': -1}

def extract_hog(img):
    return hog(img, visualize=True, **HOG_PARAMS)

def detect_edges_and_lines(img, canny_low=30, canny_high=100, hough_threshold=15,
                            min_line_length=20, max_line_gap=15, road_region_frac=0.55):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    H, W = gray.shape
    road_mask = np.zeros_like(gray); road_mask[int(H * road_region_frac):, :] = 1
    edges = cv2.Canny(gray, canny_low, canny_high)
    edges_masked = edges * road_mask
    lines_raw = cv2.HoughLinesP(edges_masked, rho=1, theta=np.pi / 180, threshold=hough_threshold,
                                 minLineLength=min_line_length, maxLineGap=max_line_gap)
    lines = []
    if lines_raw is not None:
        for line in lines_raw:
            x1, y1, x2, y2 = line[0]
            angle = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
            if 20 < angle < 75 or 105 < angle < 160:
                lines.append((x1, y1, x2, y2))
    return edges, np.array(lines)

def estimate_vanishing_point(lines, img_shape):
    H, W = img_shape[:2]
    def line_intersection(l1, l2):
        x1, y1, x2, y2 = l1; x3, y3, x4, y4 = l2
        denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
        if abs(denom) < 1e-6: return None
        t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denom
        return (x1 + t * (x2 - x1), y1 + t * (y2 - y1))
    if len(lines) < 2: return (0.5, 0.5), len(lines)
    pts = [p for i in range(len(lines)) for j in range(i + 1, len(lines))
           if (p := line_intersection(lines[i], lines[j])) and -W < p[0] < 2 * W and -H < p[1] < 2 * H]
    if not pts: return (0.5, 0.5), len(lines)
    xs, ys = np.array([p[0] for p in pts]), np.array([p[1] for p in pts])
    hist, xe, ye = np.histogram2d(xs, ys, bins=[np.linspace(-W, 2*W, 30), np.linspace(-H, 2*H, 30)])
    pi = np.unravel_index(hist.argmax(), hist.shape)
    return (((xe[pi[0]] + xe[pi[0]+1]) / 2) / W, ((ye[pi[1]] + ye[pi[1]+1]) / 2) / H), len(lines)

def get_robust_seed_color(hsv, H, W):
    pts = [(W//2, int(H*.95)), (W//2, int(H*.85)), (int(W*.35), int(H*.92)),
           (int(W*.65), int(H*.92)), (int(W*.35), int(H*.82)), (int(W*.65), int(H*.82))]
    colors, valid = [], []
    for sx, sy in pts:
        patch = hsv[max(0, sy-5):sy+5, max(0, sx-10):sx+10]
        if patch.size > 0:
            colors.append(np.mean(patch, axis=(0, 1))); valid.append((sx, sy))
    colors = np.array(colors)
    med = np.median(colors, axis=0)
    best = np.argmin(np.linalg.norm(colors - med, axis=1))
    return colors[best].astype(np.uint8), valid[best]

def segment_road(img, road_region_frac=0.55):
    H, W = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    seed_color, (sx, sy) = get_robust_seed_color(hsv, H, W)
    tol = np.array([20, 60, 60])
    mask = cv2.inRange(hsv, np.clip(seed_color.astype(int)-tol, 0, 255).astype(np.uint8),
                        np.clip(seed_color.astype(int)+tol, 0, 255).astype(np.uint8))
    mask[:int(H*road_region_frac), :] = 0
    flood = np.zeros((H+2, W+2), dtype=np.uint8)
    cv2.floodFill(mask.copy(), flood, (sx, sy), 255, loDiff=(10,10,10), upDiff=(10,10,10))
    road_mask = (flood[1:-1, 1:-1] * 255).astype(np.uint8)
    px = np.where(road_mask > 0)
    if len(px[0]) < 10:
        return road_mask, {'road_area_frac': 0., 'road_centroid_x': 0.5, 'road_taper': 0.}
    area = len(px[0]) / (H * W)
    cx = np.mean(px[1]) / W
    taper = (np.sum(road_mask[int(H*.9), :] > 0) - np.sum(road_mask[int(H*.7), :] > 0)) / W
    return road_mask, {'road_area_frac': area, 'road_centroid_x': cx, 'road_taper': taper}

yolo_model = YOLO('yolov8s.pt')

fig, axes = plt.subplots(5, len(CLASSES), figsize=(4.2 * len(CLASSES), 18))
for i, cls in enumerate(CLASSES):
    img = examples[cls]
    H, W = img.shape[:2]

    axes[0, i].imshow(img); axes[0, i].set_title(cls, fontsize=13, fontweight='bold'); axes[0, i].axis('off')

    _, hog_img = extract_hog(img)
    axes[1, i].imshow(hog_img, cmap='gray'); axes[1, i].set_title('HOG'); axes[1, i].axis('off')

    results = yolo_model(img, verbose=False)
    axes[2, i].imshow(results[0].plot()); axes[2, i].set_title('YOLO detections'); axes[2, i].axis('off')

    edges, lines = detect_edges_and_lines(img)
    vp, n_lines = estimate_vanishing_point(lines, img.shape)
    img_vp = img.copy()
    for x1, y1, x2, y2 in lines: cv2.line(img_vp, (x1, y1), (x2, y2), (0, 255, 0), 1)
    vpx = (int(vp[0]*W), int(vp[1]*H))
    if 0 <= vpx[0] < W and 0 <= vpx[1] < H: cv2.circle(img_vp, vpx, 6, (255, 0, 0), -1)
    axes[3, i].imshow(img_vp); axes[3, i].set_title(f'Hough+VP ({n_lines} lines)'); axes[3, i].axis('off')

    road_mask, feats = segment_road(img)
    overlay = img.copy()
    overlay[road_mask > 0] = (overlay[road_mask > 0]*0.5 + np.array([0,255,0])*0.5).astype(np.uint8)
    axes[4, i].imshow(overlay)
    axes[4, i].set_title(f'area={feats["road_area_frac"]:.2f} cx={feats["road_centroid_x"]:.2f}')
    axes[4, i].axis('off')

plt.suptitle('Sample Frame + Feature Visualization by Maneuver Class', fontsize=16)
plt.tight_layout()
plt.savefig('class_feature_panel_full.png', dpi=150)
plt.show()

## 2. t-SNE and UMAP alongside PCA

The existing `pca_visualization.png` cell only looks at linear structure. t-SNE and UMAP can reveal
nonlinear clusters PCA would miss — useful for sanity-checking whether a feature is *really* uninformative
for maneuver class, or just not linearly so.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import umap

hog_feats  = np.load(os.path.join(FEATURES_DIR, 'hog.npy'))
hsv_feats  = np.load(os.path.join(FEATURES_DIR, 'hsv.npy'))
yolo_feats = np.load(os.path.join(FEATURES_DIR, 'yolo.npy'))
road_feats = np.load(os.path.join(FEATURES_DIR, 'road.npy'))
road_v2_path = os.path.join(FEATURES_DIR, 'road_v2.npy')
road_v2_feats = np.load(road_v2_path) if os.path.exists(road_v2_path) else None
labels     = np.load(os.path.join(FEATURES_DIR, 'labels.npy'), allow_pickle=True)

feature_sets = {
    'HOG': hog_feats, 'HSV': hsv_feats, 'YOLO (detections)': yolo_feats,
    'Road Geometry (v1)': road_feats,
    'All Combined': np.concatenate([hog_feats, hsv_feats, yolo_feats, road_feats], axis=1),
}
if road_v2_feats is not None:
    feature_sets['Road Geometry (v2)'] = road_v2_feats  # camera-segment-restricted Hough+VP, see Section 8

# include the CNN embedding feature sets once produced (sections 3 and 6)
extra_embeddings = {
    'YOLO-CNN Embedding': 'cnn_embedding.npy',
    'MobileNetV2 Embedding': 'mobilenetv2_embedding.npy',
    'ViT Embedding': 'vit_embedding.npy',
}
vehicle_occ_path = os.path.join(FEATURES_DIR, 'vehicle_occupancy.npy')
vehicle_occ_feats = np.load(vehicle_occ_path) if os.path.exists(vehicle_occ_path) else None
if vehicle_occ_feats is not None:
    feature_sets['Vehicle Occupancy (Section 9)'] = vehicle_occ_feats
loaded_embeddings = {}
for name, fname in extra_embeddings.items():
    path_ = os.path.join(FEATURES_DIR, fname)
    if os.path.exists(path_):
        feats = np.load(path_)
        feature_sets[name] = feats
        loaded_embeddings[name] = feats

if loaded_embeddings:
    feature_sets['All + All Embeddings'] = np.concatenate(
        [hog_feats, hsv_feats, yolo_feats, road_feats] + list(loaded_embeddings.values()), axis=1)
    print(f'Loaded embeddings: {list(loaded_embeddings.keys())}')

methods = {
    'PCA': lambda X: PCA(n_components=2, random_state=0).fit_transform(X),
    't-SNE': lambda X: TSNE(n_components=2, random_state=0, perplexity=30, init='pca').fit_transform(X),
    'UMAP': lambda X: umap.UMAP(n_components=2, random_state=0).fit_transform(X),
}

fig, axes = plt.subplots(len(methods), len(feature_sets), figsize=(5*len(feature_sets), 5*len(methods)))
for row, (mname, mfunc) in enumerate(methods.items()):
    for col, (fname, feats) in enumerate(feature_sets.items()):
        ax = axes[row, col]
        X = StandardScaler().fit_transform(feats)
        proj = mfunc(X)
        for cls in CLASSES:
            mask = labels == cls
            ax.scatter(proj[mask, 0], proj[mask, 1], c=COLORS[cls], label=cls, alpha=0.4, s=8)
        if row == 0: ax.set_title(fname, fontsize=12)
        if col == 0: ax.set_ylabel(mname, fontsize=12)
        if row == 0 and col == len(feature_sets) - 1:
            ax.legend(fontsize=7, markerscale=2, loc='upper right')
        ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('PCA vs t-SNE vs UMAP by Feature Set and Maneuver Class', fontsize=15)
plt.tight_layout()
plt.savefig('embedding_comparison.png', dpi=140)
plt.show()

**Road Geometry v1 vs v2 (sandbox re-run after adding v2, cell above):** visually, v2 does **not**
show cleaner class separation than v1 in any of PCA/t-SNE/UMAP -- both project into the same
overlapping blob across all 5 maneuver classes with no visible per-class structure. The camera-stitch
fix makes the vanishing-point *geometry* correct (see Section 8), but geometric correctness alone
doesn't translate into more class-discriminative *features* -- consistent with the pattern seen
everywhere else in this notebook (HSV, YOLO, Road v1 all show the same lack of visual separability).
The SVM/RF baseline below gives a more decisive answer than eyeballing projections.

**What we're seeing (ran this on the sandbox copy of the precomputed features):** across all three
projection methods, none of HOG, HSV, YOLO-detections, or Road Geometry cleanly separates the 5 maneuver
classes — points from different classes are heavily interleaved everywhere except a few small, tight
YOLO clusters that look like they're grouping by *scene type* (lighting/object density) rather than
maneuver. Road Geometry shows the least-bad structure, consistent with the centroid-shift finding already
in the README, but it's still far from separable on its own.

Two things worth flagging for the group:

- **Class imbalance.** Straight is 52% of the data vs. 3.7% for lane-change-left. In every 2D projection,
  straight (blue) dominates by sheer point count, which can visually hide whether the minority classes
  are actually separable — worth re-plotting with balanced subsampling or per-class density contours
  instead of raw scatter alpha.
- **Single-frame limitation.** All current features come from one target frame. A "maneuver" is inherently
  a *motion* pattern — a single frame can only proxy it indirectly (vanishing point position, road
  centroid). Bringing in the past frames (already present in `context_fnames`) via optical flow or a
  simple frame-difference feature would likely help more than refining any single-frame feature further.

## 3. New complex feature: dense CNN embedding (YOLOv8s backbone)

The current YOLO feature only keeps detection counts/positions for 8 COCO classes — a sparse summary that
throws away most of what the CNN "saw". Ultralytics exposes the backbone's pooled feature vector directly
via `model.embed(...)`, which is a much denser, more general-purpose visual representation from the same
pretrained network already in the pipeline (no new model weights to source or justify).

In [ ]:
import time

EMBED_LAYER = len(yolo_model.model.model) - 2  # penultimate layer, standard choice per Ultralytics docs

t0 = time.time()
all_cnn = []
for i, (seq_id, entry) in enumerate(manifest.items()):
    if (i + 1) % 200 == 0:
        print(f'{i+1}/{len(manifest)} ({time.time()-t0:.0f}s)')
    data = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    img = np.array(data['_modality_data'].item()[Modality.CAMERAS])
    emb = yolo_model.embed(img, verbose=False)[0]
    all_cnn.append(emb.cpu().numpy())

cnn_embedding = np.array(all_cnn)
np.save(os.path.join(FEATURES_DIR, 'cnn_embedding.npy'), cnn_embedding)
print(f'CNN embedding shape: {cnn_embedding.shape}  ({time.time()-t0:.0f}s total)')
print('Re-run the Section 2 cell above -- it will pick up cnn_embedding.npy automatically.')

## 4. Summary + tinkering ideas for next iteration

**For the update:** the per-class panel above (section 1) and the PCA/t-SNE/UMAP comparison (section 2)
are the two artifacts to include. Headline finding: current single-frame features (HOG, HSV, YOLO
detections, Road Geometry) do not cleanly separate the 5 maneuver classes under any of the three
projection methods tried — this motivates trying denser/temporal features rather than tuning the
existing ones further.

**Ideas worth trying, roughly in order of expected payoff:**

1. **Bring in the past frames.** `context_fnames` already exists per sequence — a simple frame-difference
   or optical-flow-magnitude feature between the first and last context frame would directly encode motion,
   which single-frame features can only proxy indirectly.
2. **CNN embedding (this notebook, section 3).** Denser than YOLO detection counts; check its PCA/t-SNE/UMAP
   plots once extracted — if it's still not separable, that's evidence the bottleneck is single-frame-ness,
   not feature richness.
3. **Address class imbalance before drawing conclusions from separability plots** — resample or use
   class-weighted density plots so lane-change classes aren't visually swamped by "straight".
4. **Vanishing-point / road-centroid trajectory across context frames** rather than a single value — the
   README already found centroid_x is informative in a single frame; its *drift* over the context window
   is a natural, still-classical next feature.
5. **CLIP or DINOv2 embedding** as an alternative dense feature to compare against the YOLO backbone
   embedding, if compute allows — different pretraining objective (contrastive vs. detection) might
   capture different maneuver-relevant structure.
6. **Feature selection / mutual information** against the 8-dim Road Geometry features specifically, since
   that's the one set showing any structure — worth understanding which of the 8 dims is actually
   carrying signal before expanding it.

## 5. SVM / Random Forest baseline

The PCA/t-SNE/UMAP plots above only look at 2D projections, which throw away most of the
information in HOG's 5,796 dims (and 512 for the CNN embedding). The real test of whether these
features are useful is whether a classifier trained on the *full* feature vectors beats the
majority-class baseline (51.8% — always guessing "straight").

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

cnn_feats = np.load(os.path.join(FEATURES_DIR, 'cnn_embedding.npy'))
road_v2_path = os.path.join(FEATURES_DIR, 'road_v2.npy')
road_v2_feats = np.load(road_v2_path) if os.path.exists(road_v2_path) else None

feature_sets_clf = {
    'HSV': hsv_feats, 'YOLO': yolo_feats, 'Road': road_feats, 'CNN': cnn_feats,
    'HOG': hog_feats,
    'All_Combined': np.concatenate([hog_feats, hsv_feats, yolo_feats, road_feats], axis=1),
    'All_CNN': np.concatenate([hog_feats, hsv_feats, yolo_feats, road_feats, cnn_feats], axis=1),
}
if road_v2_feats is not None:
    # direct v1-vs-v2 test: same combined feature sets, v1's Road swapped for v2's Road_v2
    feature_sets_clf['Road_v2'] = road_v2_feats
    feature_sets_clf['All_Combined_v2Road'] = np.concatenate(
        [hog_feats, hsv_feats, yolo_feats, road_v2_feats], axis=1)
    feature_sets_clf['All_CNN_v2Road'] = np.concatenate(
        [hog_feats, hsv_feats, yolo_feats, road_v2_feats, cnn_feats], axis=1)

# pick up MobileNetV2 / ViT embeddings if section 6 has been run
vehicle_occ_path = os.path.join(FEATURES_DIR, 'vehicle_occupancy.npy')
vehicle_occ_feats = np.load(vehicle_occ_path) if os.path.exists(vehicle_occ_path) else None
if vehicle_occ_feats is not None:
    feature_sets_clf['VehicleOcc'] = vehicle_occ_feats
    feature_sets_clf['All_CNN_VehicleOcc'] = np.concatenate(
        [hog_feats, hsv_feats, yolo_feats, road_feats, cnn_feats, vehicle_occ_feats], axis=1)

extra_embeddings_clf = {'MNv2': 'mobilenetv2_embedding.npy', 'ViT': 'vit_embedding.npy'}
loaded_extra = {}
for short_name, fname in extra_embeddings_clf.items():
    path_ = os.path.join(FEATURES_DIR, fname)
    if os.path.exists(path_):
        feats = np.load(path_)
        feature_sets_clf[short_name] = feats
        loaded_extra[short_name] = feats

if loaded_extra:
    feature_sets_clf['All_CNN_MNv2_ViT'] = np.concatenate(
        [hog_feats, hsv_feats, yolo_feats, road_feats, cnn_feats] + list(loaded_extra.values()), axis=1)
    print(f'Loaded extra embeddings for classification: {list(loaded_extra.keys())}')

majority_baseline = (labels == 'straight').mean()
print(f'Majority-class baseline: {majority_baseline:.3f}\n')

results = {}
best = (None, None, 0.0, None, None)  # feature_set, model, acc, y_test, y_pred

for name, feats in feature_sets_clf.items():
    X_train, X_test, y_train, y_test = train_test_split(
        feats, labels, test_size=0.25, random_state=0, stratify=labels)
    scaler = StandardScaler()
    X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

    svm = SVC(kernel='rbf', class_weight='balanced', random_state=0)
    svm.fit(X_train_s, y_train)
    svm_pred = svm.predict(X_test_s)
    svm_acc = accuracy_score(y_test, svm_pred)

    rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=0, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_acc = accuracy_score(y_test, rf_pred)

    results[name] = dict(svm_acc=svm_acc, rf_acc=rf_acc, rf_importances=rf.feature_importances_,
                          y_test=y_test, svm_pred=svm_pred, rf_pred=rf_pred)
    print(f'{name:14s}  SVM: {svm_acc:.3f}   RF: {rf_acc:.3f}')
    if svm_acc > best[2]: best = (name, 'svm', svm_acc, y_test, svm_pred)
    if rf_acc > best[2]: best = (name, 'rf', rf_acc, y_test, rf_pred)

print(f'\nBest: {best[0]} / {best[1].upper()}  acc={best[2]:.3f}  (majority={majority_baseline:.3f})')

In [ ]:
# confusion matrix for the best model + accuracy comparison bar chart
name, model, acc, y_test, y_pred = best
cm = confusion_matrix(y_test, y_pred, labels=CLASSES)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix -- {name} / {model.upper()}\nacc={acc:.3f} (majority baseline={majority_baseline:.3f})')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.savefig('confusion_matrix_best.png', dpi=150)
plt.show()

print(classification_report(y_test, y_pred, labels=CLASSES, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(feature_sets_clf))
width = 0.35
ax.bar(x - width/2, [results[n]['svm_acc'] for n in feature_sets_clf], width, label='SVM')
ax.bar(x + width/2, [results[n]['rf_acc'] for n in feature_sets_clf], width, label='RF')
ax.axhline(majority_baseline, color='red', linestyle='--', label=f'Majority baseline ({majority_baseline:.3f})')
ax.set_xticks(x); ax.set_xticklabels(list(feature_sets_clf.keys()), rotation=20, ha='right')
ax.set_ylabel('Test accuracy'); ax.set_title('SVM / RF Accuracy by Feature Set vs. Majority Baseline')
ax.legend()
plt.tight_layout()
plt.savefig('baseline_accuracy_comparison.png', dpi=150)
plt.show()

road_dims = ['vp_x', 'vp_y', 'n_lines', 'road_area_frac', 'road_centroid_x', 'road_width_bottom', 'road_width_mid', 'road_taper']
print('\nRF feature importances -- Road Geometry:')
for dim, imp in sorted(zip(road_dims, results['Road']['rf_importances']), key=lambda t: -t[1]):
    print(f'  {dim:20s} {imp:.3f}')

if 'Road_v2' in results:
    road_v2_dims = ['vp_x', 'vp_y', 'n_lines', 'n_inliers', 'inlier_ratio',
                    'road_area_frac', 'road_centroid_x', 'road_width_bottom', 'road_width_mid', 'road_taper']
    print('\nRF feature importances -- Road Geometry v2:')
    for dim, imp in sorted(zip(road_v2_dims, results['Road_v2']['rf_importances']), key=lambda t: -t[1]):
        print(f'  {dim:20s} {imp:.3f}')

if 'VehicleOcc' in results:
    vehicle_occ_dims = ['left_count', 'left_crowding', 'left_nearest',
                         'right_count', 'right_crowding', 'right_nearest', 'asymmetry']
    print('\nRF feature importances -- Vehicle Occupancy (Section 9):')
    for dim, imp in sorted(zip(vehicle_occ_dims, results['VehicleOcc']['rf_importances']), key=lambda t: -t[1]):
        print(f'  {dim:20s} {imp:.3f}')

**Results (from Dawson's local run against the real feature arrays -- authoritative over the earlier
sandbox numbers below):**

| Feature set | SVM acc | RF acc |
|---|---|---|
| Majority baseline | 0.518 | 0.518 |
| HSV | 0.269 | 0.471 |
| YOLO | 0.230 | 0.441 |
| Road Geometry | 0.241 | 0.392 |
| CNN Embedding | 0.533 | 0.565 |
| HOG | 0.635 | 0.601 |
| All Combined | 0.642 | 0.606 |
| All + CNN Embedding | **0.657** | 0.612 |

SVM accuracy matches the earlier sandbox run to three decimals across every feature set -- good
confirmation that data, train/test split, and preprocessing are all consistent. RF differs meaningfully
from the sandbox run (there it looked stuck around 0.53 regardless of feature richness; here it scales
up with feature richness much like SVM does, 0.471 -> 0.606 -> 0.612). That's most likely a scikit-learn
version difference between this sandbox and the local `281-s2-group2` env -- `RandomForestClassifier`'s
reproducibility guarantee under a fixed `random_state` is version-specific, unlike SVM's solver. Trust
these numbers over the earlier sandbox ones; the corrected takeaway is that RF is a perfectly reasonable
second model here, not a weak one -- it just needed the real environment to show it.

So the earlier pessimism about the projection plots wasn't warranted: HOG and the CNN embedding both
clearly beat the majority baseline once a classifier sees the full feature vector, and combining
everything (SVM on All + CNN Embedding) gets to 65.7%, +14 points over baseline. RF landing at 61.2% on
the same combined feature set is a reasonable second option to keep around, e.g. for its feature
importances and lower variance.

**The confusion matrix confirms the imbalance problem exactly as suspected:** straight and
right-turn are learned reasonably well (recall 0.84 and 0.69), but lane-change-left/right are
almost always predicted as straight (13/17 and 15/22 misclassified that way) despite
`class_weight='balanced'`. With only 17-22 test examples per minority class, the model doesn't
have enough signal to learn them. Precision is 1.00 on the rare occasions it does predict a
lane-change, so the model isn't confusing lane-changes with turns -- it's just not confident
enough to predict them at all. This is the concrete next thing to address: oversampling
(SMOTE), a class-weighted loss tuned more aggressively, or just collecting more lane-change
examples, before more feature engineering effort.

**Road Geometry importances also confirm the README's centroid-shift claim quantitatively:**
`road_centroid_x` is tied for the top RF feature importance among the 8 Road Geometry dims,
not just a qualitative pattern in a PCA plot.

**Road Geometry v1 vs v2 -- SVM/RF baseline (Dawson's local run, authoritative):**

| Feature set | SVM acc | RF acc |
|---|---|---|
| Road (v1, 8d) | 0.241 | 0.392 |
| Road_v2 (10d) | 0.222 | 0.437 |
| All_Combined (v1 road) | 0.642 | 0.606 |
| All_Combined_v2Road | 0.642 | 0.599 |
| All_CNN (v1 road) | 0.657 | 0.612 |
| All_CNN_v2Road | 0.657 | 0.586 |
| Best overall | **All_CNN / SVM, acc=0.657** (unchanged by road version) | |

**SVM is completely indifferent to which road feature you use** -- 0.642 and 0.657 match to 3
decimals whether it's fed v1 or v2's road dims, because the RBF kernel weighs all input dims jointly
and the road feature (8 or 10 of 5,860+ combined dims) barely registers either way.

**RF tells a more interesting, two-sided story.** Standalone, v2 actually beats v1 by a real margin
(0.437 vs 0.392, +0.045) -- `n_lines`/`n_inliers`/`inlier_ratio` apparently give RF's tree splits more
to work with than v1's flatter 8-dim vector. But swapped into the full combined feature set, v2 makes
RF *worse* (All_Combined: 0.606 -> 0.599; All_CNN: 0.612 -> 0.586, the largest drop in the table). The
likely mechanism: RF's default `max_features='sqrt'` spreads split candidates across all input dims,
and road_v2's 2 extra, partially-redundant dims (`inlier_ratio` correlates with `n_inliers`/`n_lines`)
dilute the sampling probability of the road-segmentation dims (`road_centroid_x`, `road_area_frac`)
that were actually carrying the signal in v1 -- more dimensions from the same feature family without
proportionally more information, which mildly hurts a bagged tree ensemble even though it doesn't
touch a kernel method at all.

**Practical takeaway: use v1's Road Geometry (8d) in the final combined feature set, not v2.** v2 is
the scientifically correct version (Section 8's vanishing points are the ones that are actually
geometrically valid) and worth keeping in the writeup as the "fixed" version, but it isn't the version
that should feed the classifier -- it's marginally better for RF in isolation and net-negative for RF
combined, with SVM unaffected either way. This closes the loop from the "are we fucked" question:
the Hough+VP bug was real and worth fixing for correctness, but it was never the accuracy bottleneck,
and fixing it doesn't move the 5-class separability ceiling either direction in any way that matters.

## 6. Additional pretrained-CNN feature sets: MobileNetV2 and ViT

The literature review (Djuric et al. 2020, Table 1; Chou et al. 2020, Table II) found MobileNetV2 to be
the strongest base CNN among AlexNet/VGG-19/ResNet-50/MobileNetV2/MnasNet for raster-based motion
prediction -- and it's already available via `torchvision`, no new install needed (unlike `ultralytics`).

We also add a plain supervised ViT-B/16 (same ImageNet pretraining data/objective as MobileNetV2, only
the architecture changes -- local convolutions vs. global self-attention). This isolates architecture as
the variable, rather than conflating it with a different training paradigm (as CLIP or DINO would).
ViT expects roughly square 224x224 input, so the 142x384 panorama needs resizing -- we center-crop-and-pad
rather than squash, to avoid warping the horizontal geometry (vanishing point, road curvature) the whole
task depends on.

### What MobileNetV2 and ViT actually are

**MobileNetV2** (Sandler et al., 2018) is a CNN designed for efficient inference on phones/embedded
devices, and it's the architecture that came out on top in the motion-prediction literature we reviewed
(Djuric et al. 2020, Table 1). Its core building block is the **inverted residual with linear
bottleneck**: a 1x1 convolution *expands* the number of channels, a 3x3 **depthwise separable**
convolution processes each channel independently (spatially), and another 1x1 convolution *projects*
back down to a small number of channels, with a residual (skip) connection linking input to output. This
is the reverse of a classic ResNet block (which goes wide -> narrow -> wide) -- hence "inverted." The
depthwise separable convolution is the key efficiency trick: a standard convolution mixes space and
channels together in one expensive operation, while depthwise separable splits that into a cheap
per-channel spatial pass followed by a cheap 1x1 channel-mixing pass, cutting compute by roughly an
order of magnitude for similar accuracy. Like all CNNs, it builds up a global understanding of the image
gradually -- each layer only sees a local neighborhood, and larger patterns emerge by stacking layers.
It's pretrained on ImageNet (1.2M images, 1,000 object classes). In our pipeline we strip its final
classification layer and global-average-pool the last convolutional feature map, giving a 1,280-dim
vector per image that summarizes what the network "saw," the same way we already pulled a feature vector
out of the YOLO backbone.

**ViT (Vision Transformer)** (Dosovitskiy et al., 2020, "An Image is Worth 16x16 Words") takes a very
different approach: instead of convolutions, it applies the Transformer architecture originally built for
language directly to images. The image is cut into a grid of fixed-size patches (16x16 pixels for the
ViT-B/16 variant we use), each patch is flattened and linearly projected into a vector, a position
embedding is added so the model knows where each patch came from, and a learnable **[CLS] token** is
prepended to the sequence. All of this is fed through standard Transformer encoder layers, where
**self-attention** lets every patch directly attend to every other patch, regardless of distance --
so the model can relate the left and right edges of an image in its very first layer, something a CNN
only achieves after enough stacked layers to grow its receptive field that wide. After all the encoder
layers, the [CLS] token's final representation is treated as a whole-image summary -- that's the 768-dim
vector we extract per image. Like MobileNetV2, it's pretrained on ImageNet for classification; the only
thing that differs between the two feature sets we're comparing is architecture (convolution vs.
attention), not training data or objective.

| | MobileNetV2 | ViT-B/16 |
|---|---|---|
| Core operation | Depthwise separable convolution | Self-attention over image patches |
| How it builds context | Gradually, layer by layer (local -> global) | Directly, from the first layer (global from the start) |
| Pretraining | ImageNet-1k classification | ImageNet-1k classification |
| Embedding we extract | 1,280-dim (pooled last conv layer) | 768-dim (CLS token, pre-logits) |
| Why we're trying it | Best base CNN in the reviewed literature; already in `torchvision` | Tests whether global attention captures maneuver-relevant structure (e.g., vanishing point vs. image edges) that local convolutions miss |

The reason both are worth trying alongside the existing YOLO backbone embedding: YOLO's backbone was
trained for *detection* (find and localize specific objects), while MobileNetV2 and ViT were trained for
*classification* (what's the single dominant thing in this image) -- three different pretraining
objectives and, for ViT, a fundamentally different architecture, all applied to the same question of
whether a pretrained network's internal representation happens to be sensitive to what distinguishes a
straight frame from a turning one.

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights, vit_b_16, ViT_B_16_Weights
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__}, torchvision {torchvision.__version__}, device={device}')

# --- MobileNetV2: strip classifier head, global-average-pool the last conv feature map ---
mnv2 = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1).to(device).eval()
mnv2_preprocess = MobileNet_V2_Weights.IMAGENET1K_V1.transforms()

def extract_mobilenetv2(img):
    pil_img = Image.fromarray(img)
    x = mnv2_preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        feats = mnv2.features(x)               # (1, 1280, H', W')
        pooled = feats.mean(dim=[2, 3])          # global average pool -> (1, 1280)
    return pooled.squeeze(0).cpu().numpy()

# --- ViT-B/16: use the pre-logits CLS token as the embedding ---
vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1).to(device).eval()

def pad_to_square(pil_img):
    # center-crop-and-pad instead of squashing, to preserve the horizontal geometry
    w, h = pil_img.size
    size = max(w, h)
    new_img = Image.new('RGB', (size, size), (0, 0, 0))
    new_img.paste(pil_img, ((size - w) // 2, (size - h) // 2))
    return new_img

vit_preprocess = transforms.Compose([
    transforms.Lambda(pad_to_square),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_vit(img):
    pil_img = Image.fromarray(img)
    x = vit_preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        # torchvision's ViT forward up to (but not including) the classification head:
        feats = vit._process_input(x)
        n = feats.shape[0]
        batch_class_token = vit.class_token.expand(n, -1, -1)
        feats = torch.cat([batch_class_token, feats], dim=1)
        feats = vit.encoder(feats)
        cls_token = feats[:, 0]                  # (1, 768) pre-logits CLS embedding
    return cls_token.squeeze(0).cpu().numpy()

print('MobileNetV2 and ViT-B/16 loaded and ready.')

In [ ]:
import time

t0 = time.time()
all_mnv2, all_vit = [], []
for i, (seq_id, entry) in enumerate(manifest.items()):
    if (i + 1) % 200 == 0:
        print(f'{i+1}/{len(manifest)} ({time.time()-t0:.0f}s)')
    data = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    img = np.array(data['_modality_data'].item()[Modality.CAMERAS])
    all_mnv2.append(extract_mobilenetv2(img))
    all_vit.append(extract_vit(img))

mnv2_embedding = np.array(all_mnv2)
vit_embedding = np.array(all_vit)
np.save(os.path.join(FEATURES_DIR, 'mobilenetv2_embedding.npy'), mnv2_embedding)
np.save(os.path.join(FEATURES_DIR, 'vit_embedding.npy'), vit_embedding)
print(f'MobileNetV2 embedding: {mnv2_embedding.shape}')
print(f'ViT embedding: {vit_embedding.shape}')
print(f'Total time: {time.time()-t0:.0f}s')
print('Re-run the Section 2 / Section 5 cells above -- they will pick these up automatically once you add them to feature_sets.')

**MobileNetV2 / ViT baseline results (Dawson's local run):**

| Feature set | SVM acc | RF acc |
|---|---|---|
| CNN (YOLOv8s backbone) | 0.533 | 0.565 |
| MobileNetV2 | 0.571 | 0.597 |
| ViT-B/16 | 0.535 | 0.559 |
| All_CNN_MNv2_ViT (everything combined) | 0.657 | 0.608 |

Answering the three questions posed above, now with real numbers:

- **ViT's projection plots (previous message) showed the same non-class-related bimodal split as the
  other two CNN embeddings** -- no evidence that global attention captures anything about maneuver that
  local convolutions miss. Standalone accuracy backs this up: ViT (0.535/0.559) is statistically tied
  with the YOLO backbone (0.533/0.565), not meaningfully better.
- **MobileNetV2 (classification-pretrained) actually beats the YOLO backbone (detection-pretrained)**
  standalone -- 0.571/0.597 vs 0.533/0.565, the best of the three individual CNN embeddings. Weak
  evidence that a classification objective transfers slightly better than a detection objective for a
  whole-scene task like this, though the gap is small relative to the overall noise in this dataset size.
- **Combined, all three pretrained embeddings together (`All_CNN_MNv2_ViT`) tie the existing ceiling
  exactly** -- SVM 0.657, identical to `All_CNN` with just the YOLO embedding, and RF at 0.608 is
  actually slightly *below* `All_CNN`'s 0.612. Adding two more pretrained architectures contributes
  nothing beyond what YOLO's embedding alone already gave the classifier.

**This is now a four-for-four result: YOLO backbone, MobileNetV2, ViT, and all three combined all
plateau at the same ~0.65-0.66 wall**, regardless of architecture, pretraining objective (detection vs.
classification vs. no strong inductive bias at all for ViT), or how many of them you stack together.
That's a much stronger version of the earlier conclusion: this isn't "we haven't found the right
pretrained network yet" -- it's consistent with a genuine single-frame information ceiling for this
5-class task, independent of feature engineering effort or model architecture.

## 7. Addressing the lane-change imbalance: SMOTE vs. duplication+jitter

The confusion matrix in Section 5 showed lane-change-left/right are almost always predicted as
"straight" (13/17 and 15/22 misclassified that way), despite `class_weight='balanced'`. Two standard
fixes for class imbalance, both applied to the training split only (the test set is never touched, so
accuracy stays comparable to the Section 5 baseline):

- **SMOTE** synthesizes new minority-class examples by interpolating between real ones and their nearest
  neighbors in feature space.
- **Duplication + jitter** is the simpler alternative: copy real minority examples and add small Gaussian
  noise (scaled to each feature's std) so copies aren't exact duplicates.

Both are applied only to `lane-change-left`/`lane-change-right`, targeting 3x their original training
count (159 and 192 examples respectively, up from 53 and 64) -- not a full balance to the majority class,
since interpolating/jittering from only ~53-64 real examples up to 728 would mean most of the "signal"
in the oversampled data is synthetic repetition of a tiny sample, not new information.

**Important methodological note:** `class_weight='balanced'` is kept in *every* condition below, including
the resampled ones. An earlier version of this experiment turned it off for the resampled runs to avoid
"double-compensating" for the imbalance -- that was a mistake: 3x oversampling alone is a much weaker
correction than balanced class weighting, so removing the weighting just made every condition worse
without isolating anything about SMOTE or duplication specifically. Worth remembering if you extend this:
resampling and class weighting are not alternatives to choose between, they compound.

In [ ]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

MINORITY_CLASSES = ['lane-change-left', 'lane-change-right']

# reuse the "All_CNN" feature set (current best: HOG + HSV + YOLO + Road + YOLO-CNN embedding)
feats = np.concatenate([hog_feats, hsv_feats, yolo_feats, road_feats, cnn_feats], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    feats, labels, test_size=0.25, random_state=0, stratify=labels)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
train_counts = Counter(y_train)
print(f'Training class counts: {train_counts}')


def duplicate_with_jitter(X, y, target_classes, multiplier=3, jitter_frac=0.1, random_state=0):
    """Duplicate each target class `multiplier`x, adding Gaussian noise scaled to
    each feature's std so copies aren't exact duplicates."""
    rng = np.random.RandomState(random_state)
    feature_std = X.std(axis=0)
    feature_std[feature_std == 0] = 1.0
    X_aug, y_aug = [X], [y]
    for cls in target_classes:
        mask = y == cls
        X_cls = X[mask]
        for _ in range(multiplier - 1):
            noise = rng.normal(0, jitter_frac, size=X_cls.shape) * feature_std
            X_aug.append(X_cls + noise)
            y_aug.append(np.array([cls] * len(X_cls)))
    return np.concatenate(X_aug), np.concatenate(y_aug)


def evaluate(name, X_tr, y_tr):
    svm = SVC(kernel='rbf', class_weight='balanced', random_state=0)
    svm.fit(X_tr, y_tr)
    pred = svm.predict(X_test_s)
    acc = accuracy_score(y_test, pred)
    report = classification_report(y_test, pred, labels=CLASSES, zero_division=0, output_dict=True)
    lc_left_recall = report['lane-change-left']['recall']
    lc_right_recall = report['lane-change-right']['recall']
    print(f'\n=== {name} ===')
    print(f'Overall accuracy: {acc:.3f}   lc-left recall: {lc_left_recall:.2f}   lc-right recall: {lc_right_recall:.2f}')
    print(classification_report(y_test, pred, labels=CLASSES, zero_division=0))
    return dict(name=name, acc=acc, lc_left_recall=lc_left_recall, lc_right_recall=lc_right_recall)


results = []
results.append(evaluate('Baseline (class_weight=balanced)', X_train_s, y_train))

sampling_strategy = {cls: train_counts[cls] * 3 for cls in MINORITY_CLASSES}
smote = SMOTE(sampling_strategy=sampling_strategy, k_neighbors=5, random_state=0)
X_smote, y_smote = smote.fit_resample(X_train_s, y_train)
results.append(evaluate('SMOTE (3x lane-change classes)', X_smote, y_smote))

X_dup, y_dup = duplicate_with_jitter(X_train_s, y_train, MINORITY_CLASSES, multiplier=3, jitter_frac=0.1)
results.append(evaluate('Duplication + jitter (3x lane-change classes)', X_dup, y_dup))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = [r['name'].split(' (')[0] for r in results]
axes[0].bar(names, [r['acc'] for r in results], color=['steelblue', 'tomato', 'green'])
axes[0].axhline((labels == 'straight').mean(), color='gray', linestyle='--', label='Majority baseline')
axes[0].set_ylabel('Overall test accuracy'); axes[0].set_title('Overall Accuracy'); axes[0].legend()
axes[0].tick_params(axis='x', rotation=20)

x = np.arange(len(results))
width = 0.35
axes[1].bar(x - width/2, [r['lc_left_recall'] for r in results], width, label='lane-change-left recall')
axes[1].bar(x + width/2, [r['lc_right_recall'] for r in results], width, label='lane-change-right recall')
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=20)
axes[1].set_ylabel('Recall'); axes[1].set_title('Minority-Class Recall'); axes[1].legend()

plt.tight_layout()
plt.savefig('imbalance_comparison.png', dpi=150)
plt.show()

**Result: neither technique moved the needle at all.** Overall accuracy is flat across all three
conditions (0.657 / 0.657 / 0.659), and lane-change recall is bit-for-bit identical -- 0.06 for
lane-change-left, 0.09 for lane-change-right -- whether we use the original imbalanced training data,
3x SMOTE, or 3x duplication+jitter. A separate check at 8x SMOTE oversampling made things worse (left-turn
recall dropped from 0.33 to 0.19) without improving lane-change recall either.

This is a more useful result than a small improvement would have been. Oversampling techniques work by
giving the classifier more examples *within the same region of feature space* the minority class already
occupies -- they help when a class is genuinely separable but under-sampled. Getting exactly zero
movement, regardless of technique or oversampling ratio, is evidence that these particular lane-change
examples aren't occupying a distinguishable region of this ~6,300-dim feature space at all -- they're
sitting inside the "straight" and "turn" regions, not near the boundary of an under-represented cluster.
More synthetic copies of the same ~53-64 real examples can't manufacture separability that isn't already
there.

That converges with the PCA/t-SNE/UMAP finding from Section 2: this isn't (only) a data-quantity problem
for the minority classes, it's a feature-information problem across the board. The next lever to pull is
richer or different signal -- the MobileNetV2/ViT embeddings from Section 6, or eventually the
motion-based features noted as v2 work -- rather than further tuning of how we sample or weight the
data we already have.

## 8. Road Geometry v2: fixing the Hough+VP camera-stitch bug

The Hough+VP panel in Section 1 (`detect_edges_and_lines` / `estimate_vanishing_point`, cell above)
runs on the **full 384px-wide panorama**. Column-wise pixel-discontinuity analysis shows sharp seams
at `x=127->128` and `x=255->256` in every example -- the "panorama" is actually a stitch of three
128px camera views (almost certainly front-left / front-center / front-right). Feeding Hough lines
from three unrelated camera perspectives into one vanishing-point estimate is geometrically
meaningless, which is why some v1 vanishing points landed outside the image entirely (e.g. x=1.02,
x=-0.12 in normalized coordinates).

**v2 design, informed by Jiang, Gao & Xu (2010), "Computer Vision-Based Multiple-Lane Detection on
Straight Road and in a Curve":**

1. **Front-center camera segment only** (columns 128:256). Restricting to a single, geometrically
   coherent camera view is the precondition for "vanishing point" to mean anything.
2. **Two-pass, VP-informed adaptive ROI**, following the paper's preprocessing step: get a coarse VP
   estimate from a generous fixed ROI (bottom 55%, same starting guess as v1), then use that estimate
   to inform a refined ROI for a second pass. The adaptation is constrained to be one-directional --
   it can only grow the ROI upward toward a higher horizon, never shrink it -- since an unconstrained
   version compounds a noisy first-pass estimate into an even worse second pass on these short/wide
   142x384 frames.
3. **Inlier-consistency filtering**: keep only lines whose pairwise intersections land near the final
   VP estimate, discarding lines from tree branches, power lines, and building edges that survive the
   ROI mask but don't actually converge on the road's vanishing point. This is a simplified stand-in
   for the paper's left/right corner-radiating line partition, without requiring the camera calibration
   parameters that approach depends on.

Produces a 10-dim feature per sequence (vs. 8-dim in v1):
`[vp_x, vp_y, n_lines, n_inliers, inlier_ratio, road_area_frac, road_centroid_x, road_width_bottom, road_width_mid, road_taper]`
(the road-segmentation dims are unchanged from v1 -- only the Hough/VP half is new).

In [ ]:
CENTER_SEG = (128, 256)  # front-center camera segment, confirmed via seam detection above

def _line_intersection(l1, l2):
    x1, y1, x2, y2 = l1
    x3, y3, x4, y4 = l2
    denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if abs(denom) < 1e-6:
        return None
    t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denom
    return (x1 + t * (x2 - x1), y1 + t * (y2 - y1))

def _detect_lines_in_roi(gray_center, roi_top_row, canny_low=30, canny_high=100,
                          hough_threshold=12, min_line_length=15, max_line_gap=15):
    H, W = gray_center.shape
    mask = np.zeros_like(gray_center); mask[roi_top_row:, :] = 1
    edges = cv2.Canny(gray_center, canny_low, canny_high)
    edges_masked = edges * mask
    lines_raw = cv2.HoughLinesP(edges_masked, rho=1, theta=np.pi / 180, threshold=hough_threshold,
                                 minLineLength=min_line_length, maxLineGap=max_line_gap)
    lines = []
    if lines_raw is not None:
        for line in lines_raw:
            x1, y1, x2, y2 = line[0]
            angle = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
            if 20 < angle < 75 or 105 < angle < 160:
                lines.append((x1, y1, x2, y2))
    return np.array(lines)

def _estimate_vp(lines, W, H):
    fallback = (W / 2, H / 2)  # pixel-space center; caller normalizes by W/H
    if len(lines) < 2:
        return fallback, []
    intersections, pair_idx = [], []
    for i in range(len(lines)):
        for j in range(i + 1, len(lines)):
            pt = _line_intersection(lines[i], lines[j])
            if pt is not None:
                x, y = pt
                if -W < x < 2 * W and -H < y < 2 * H:
                    intersections.append((x, y)); pair_idx.append((i, j))
    if not intersections:
        return fallback, []
    xs = np.array([p[0] for p in intersections]); ys = np.array([p[1] for p in intersections])
    hist, xe, ye = np.histogram2d(xs, ys, bins=[np.linspace(-W, 2*W, 30), np.linspace(-H, 2*H, 30)])
    pi = np.unravel_index(hist.argmax(), hist.shape)
    vp_x = (xe[pi[0]] + xe[pi[0]+1]) / 2
    vp_y = (ye[pi[1]] + ye[pi[1]+1]) / 2
    return (vp_x, vp_y), list(zip(intersections, pair_idx))

def extract_road_v2(img, inlier_tol_frac=0.15):
    x0, x1 = CENTER_SEG
    center_img = img[:, x0:x1]
    gray = cv2.cvtColor(center_img, cv2.COLOR_RGB2GRAY)
    H, W = gray.shape

    # pass 1: coarse VP estimate, generous fixed ROI (bottom 55%), same starting point as v1
    lines_1 = _detect_lines_in_roi(gray, roi_top_row=int(H * 0.55))
    (vp1_x, vp1_y), _ = _estimate_vp(lines_1, W, H)

    # pass 2: refine ROI using the coarse VP -- but only ever let this GROW the search region
    # relative to the fixed 0.55 fraction, never shrink it (unconditional shrinking compounds a
    # noisy pass-1 estimate into an even worse pass-2 ROI on these short/wide frames)
    margin = int(0.15 * H)
    fixed_top_row = int(H * 0.55)
    vp_informed_top_row = int(np.clip(vp1_y - margin, 0, H - 5))
    roi_top_row_2 = min(fixed_top_row, vp_informed_top_row)
    lines_2 = _detect_lines_in_roi(gray, roi_top_row=roi_top_row_2)
    (vp2_x, vp2_y), intersections_with_idx = _estimate_vp(lines_2, W, H)

    # inlier filtering: keep lines whose intersection with at least one other line lands within
    # inlier_tol_frac of the image diagonal from the final VP estimate
    tol = inlier_tol_frac * np.hypot(W, H)
    inlier_line_idx = set()
    for (ix, iy), (i, j) in intersections_with_idx:
        if np.hypot(ix - vp2_x, iy - vp2_y) < tol:
            inlier_line_idx.add(i); inlier_line_idx.add(j)
    n_lines = len(lines_2)
    n_inliers = len(inlier_line_idx)
    inlier_ratio = n_inliers / n_lines if n_lines > 0 else 0.0

    vp_norm = (vp2_x / W, vp2_y / H)
    return vp_norm, n_lines, n_inliers, inlier_ratio, lines_2, inlier_line_idx, roi_top_row_2

def build_road_v2_features(img):
    vp, n_lines, n_inliers, inlier_ratio, *_ = extract_road_v2(img)
    road_mask, road_feats = segment_road(img)  # segment_road() defined in Section 1's cell above
    H, W = img.shape[:2]
    wb = np.sum(road_mask[int(H*.9), :] > 0) / W
    wm = np.sum(road_mask[int(H*.7), :] > 0) / W
    return np.array([
        vp[0], vp[1], n_lines, n_inliers, inlier_ratio,
        road_feats['road_area_frac'], road_feats['road_centroid_x'], wb, wm,
        road_feats['road_taper'],
    ], dtype=np.float32)

In [ ]:
fig, axes = plt.subplots(1, len(CLASSES), figsize=(4.2 * len(CLASSES), 4.5))
for i, cls in enumerate(CLASSES):
    img = examples[cls]
    H, W = img.shape[:2]
    x0, x1 = CENTER_SEG
    vp, n_lines, n_inliers, inlier_ratio, lines, inlier_idx, roi_row = extract_road_v2(img)

    img_vis = img.copy()
    for k, (lx1, ly1, lx2, ly2) in enumerate(lines):
        color = (0, 255, 0) if k in inlier_idx else (255, 165, 0)  # green=inlier, orange=rejected
        cv2.line(img_vis, (int(lx1) + x0, int(ly1)), (int(lx2) + x0, int(ly2)), color, 1)
    cv2.line(img_vis, (x0, roi_row), (x1, roi_row), (0, 200, 255), 1)  # adaptive ROI boundary
    vp_px = (int(vp[0] * (x1 - x0)) + x0, int(vp[1] * H))
    if 0 <= vp_px[0] < W and 0 <= vp_px[1] < H:
        cv2.circle(img_vis, vp_px, 6, (255, 0, 0), -1)

    axes[i].imshow(img_vis)
    axes[i].set_title(f'{cls}\nvp=({vp[0]:.2f},{vp[1]:.2f})\n{n_inliers}/{n_lines} inlier lines', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Road Geometry v2: adaptive ROI (cyan line) + inlier filtering (green=kept, orange=rejected)', fontsize=13)
plt.tight_layout()
plt.savefig('road_v2_visualization.png', dpi=140)
plt.show()

**Visual check (sandbox run on the 5 class examples):** vanishing points now land within or just at
the image bounds for every example (vp_x roughly in [0.2, 1.0], vp_y roughly in [0.6, 0.9]) -- a real
improvement over v1, which occasionally put the VP outside the frame entirely. Inlier filtering is
doing real work: right-turn keeps 18/18 lines (clean scene, few distractors) while lane-change-left
drops to 9/16 (rejects power lines/tree branches that don't converge on the road's VP). One artifact
remains -- the `straight` example's vp_x=1.02 sits just outside the segment's right edge, likely a
building edge near the seam that survives the angle filter -- worth a tighter angle threshold in a
future pass, but it doesn't change the quantitative results below.

In [ ]:
t0 = time.time()
all_road_v2 = []
for i, (seq_id, entry) in enumerate(manifest.items()):
    if (i + 1) % 200 == 0:
        print(f'{i+1}/{len(manifest)} ({time.time()-t0:.0f}s)')
    data = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    img = np.array(data['_modality_data'].item()[Modality.CAMERAS])
    all_road_v2.append(build_road_v2_features(img))

road_v2_feats = np.array(all_road_v2)
np.save(os.path.join(FEATURES_DIR, 'road_v2.npy'), road_v2_feats)
print(f'road_v2 shape: {road_v2_feats.shape}  ({time.time()-t0:.0f}s total)')
print('Re-run the Section 2 and Section 5 cells above -- they will pick up road_v2.npy automatically.')

## 9. Vehicle-position lane occupancy (adjacent-lane crowding from YOLO detections)

Idea: other vehicles (parked or moving) trace out lane structure even in a single frame, and more
directly answer "how crowded is the lane I might move into" than lane-marking geometry does.

**Why this is built as an occupancy score, not a second Hough+VP fit on detection points:** checked
first (`yolo.npy`, already computed) how many cars/trucks/buses are actually in frame across the
dataset -- median is 3, but 21% of frames have zero and only 52% have >=3, the bare minimum for a
stable line fit. Lane-change-left/right actually have the best coverage (mean 4.3-4.4 vehicles, 54-60%
with >=3), but a full line-geometry approach would still be undefined for roughly half the dataset.
Occupancy doesn't have that problem -- even a single nearby vehicle is informative about whether a lane
is open, so it degrades gracefully instead of failing outright.

**Design:**
- Restrict to the front-center camera segment (cols 128:256), same fix as Section 8 -- YOLO runs on
  the full stitched panorama, and a car detected in the left or right camera segment is a different
  camera's perspective, often a side street or perpendicular parking, not a lane the ego vehicle could
  actually merge into.
- Split each vehicle detection into left-third / right-third of that segment by its lateral offset from
  center, and weight each by `bottom_y` (bbox bottom edge / frame height) as a distance proxy -- closer
  vehicles count more toward "this lane is occupied" than distant ones.
- Produces a 7-dim feature: `[left_count, left_crowding, left_nearest, right_count, right_crowding,
  right_nearest, asymmetry]`, where `crowding` is the summed distance-weight per side and `asymmetry`
  is `left_crowding - right_crowding`.

**Note:** this needs real per-detection YOLO output (not the per-class *mean* position already saved in
`yolo.npy`, which collapses multiple vehicles into one blurry average -- the same aggregation problem
the Google Doc's feature-improvement ideas called out). Requires `ultralytics`/`torch`, same as Section
3's CNN embedding and Section 6's MobileNetV2/ViT cells -- run this locally, not in a constrained
sandbox.

In [ ]:
VEHICLE_CLASSES = {2: 'car', 5: 'bus', 7: 'truck'}  # subset of DRIVING_CLASSES relevant to lane occupancy
OCC_CENTER_SEG = (128, 256)  # front-center camera segment, same as Section 8

def extract_vehicle_occupancy(img):
    H, W = img.shape[:2]
    x0, x1 = OCC_CENTER_SEG
    seg_w = x1 - x0
    results = yolo_model(img, verbose=False)

    left = []   # list of (weight,) for vehicles in the left third of the segment
    right = []  # list of (weight,) for vehicles in the right third of the segment

    for box in results[0].boxes:
        cls_id = int(box.cls)
        if cls_id not in VEHICLE_CLASSES:
            continue
        bx1, by1, bx2, by2 = box.xyxy[0].tolist()
        cx = (bx1 + bx2) / 2
        if not (x0 <= cx < x1):
            continue  # outside the front-center segment -- different camera, skip
        rel_x = (cx - x0) / seg_w      # 0..1 within the segment
        dx = rel_x - 0.5               # -0.5 (left edge) .. +0.5 (right edge)
        bottom_y = by2 / H             # distance proxy: larger = closer to the vehicle
        weight = bottom_y

        if dx < -1/6:
            left.append(weight)
        elif dx > 1/6:
            right.append(weight)
        # detections within +/-1/6 of center are the ego's own lane, not an adjacent one -- excluded,
        # since the question this feature answers is "can I move into the lane beside me"

    left_count, right_count = len(left), len(right)
    left_crowding = sum(left) if left else 0.0
    right_crowding = sum(right) if right else 0.0
    left_nearest = max(left) if left else 0.0
    right_nearest = max(right) if right else 0.0
    asymmetry = left_crowding - right_crowding

    return np.array([left_count, left_crowding, left_nearest,
                      right_count, right_crowding, right_nearest, asymmetry], dtype=np.float32)

In [ ]:
fig, axes = plt.subplots(1, len(CLASSES), figsize=(4.2 * len(CLASSES), 4.5))
for i, cls in enumerate(CLASSES):
    img = examples[cls]
    H, W = img.shape[:2]
    x0, x1 = OCC_CENTER_SEG
    feat = extract_vehicle_occupancy(img)
    left_count, left_crowding, left_nearest, right_count, right_crowding, right_nearest, asymmetry = feat

    results = yolo_model(img, verbose=False)
    img_vis = img.copy()
    for box in results[0].boxes:
        cls_id = int(box.cls)
        if cls_id not in VEHICLE_CLASSES:
            continue
        bx1, by1, bx2, by2 = [int(v) for v in box.xyxy[0].tolist()]
        cx = (bx1 + bx2) / 2
        if not (x0 <= cx < x1):
            color = (128, 128, 128)  # outside front-center segment -- excluded, drawn gray
        else:
            dx = (cx - x0) / (x1 - x0) - 0.5
            color = (0, 255, 0) if dx < -1/6 else ((255, 0, 0) if dx > 1/6 else (255, 255, 0))
        cv2.rectangle(img_vis, (bx1, by1), (bx2, by2), color, 2)
    cv2.line(img_vis, (x0, 0), (x0, H), (0, 200, 255), 1)
    cv2.line(img_vis, (x1, 0), (x1, H), (0, 200, 255), 1)

    axes[i].imshow(img_vis)
    axes[i].set_title(f'{cls}\nL: n={int(left_count)} crowd={left_crowding:.2f}\n'
                       f'R: n={int(right_count)} crowd={right_crowding:.2f}  asym={asymmetry:+.2f}',
                       fontsize=8)
    axes[i].axis('off')

plt.suptitle('Vehicle occupancy: green=left-lane vehicle, red=right-lane vehicle, yellow=ego lane (excluded), gray=outside front-center segment', fontsize=11)
plt.tight_layout()
plt.savefig('vehicle_occupancy_visualization.png', dpi=140)
plt.show()

**What to check once this runs:** whether `left_crowding`/`right_crowding`/`asymmetry` separate the
lane-change classes from the rest in the Section 2 projections, and whether adding this 7-dim feature
moves the SVM/RF baseline in Section 5 beyond the existing 65.7% ceiling -- this is a genuinely
different information source (other agents' positions) rather than another view of road geometry, so
it's one of the more legitimate candidates to actually break the plateau, if anything is going to.

In [ ]:
t0 = time.time()
all_occ = []
for i, (seq_id, entry) in enumerate(manifest.items()):
    if (i + 1) % 200 == 0:
        print(f'{i+1}/{len(manifest)} ({time.time()-t0:.0f}s)')
    data = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    img = np.array(data['_modality_data'].item()[Modality.CAMERAS])
    all_occ.append(extract_vehicle_occupancy(img))

vehicle_occupancy = np.array(all_occ)
np.save(os.path.join(FEATURES_DIR, 'vehicle_occupancy.npy'), vehicle_occupancy)
print(f'vehicle_occupancy shape: {vehicle_occupancy.shape}  ({time.time()-t0:.0f}s total)')
print('Re-run the Section 2 / Section 5 cells above -- they will pick up vehicle_occupancy.npy automatically.')